# Testing Sandbox
Use this notebook to test resume + cover letter generation from candidate data and a job description.

In [1]:
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

# Resolve repo root regardless of where the notebook kernel starts
repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
project_dir = repo_root / "HireMe.AI-V1"

# Make modules in HireMe.AI-V1 importable
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

# Load environment variables from repo .env
load_dotenv(repo_root / ".env", override=True)

print("Repo root:", repo_root)
print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))
print("OPENAI_MODEL_RESUME:", os.getenv("OPENAI_MODEL_RESUME", "gpt-5-nano"))
print("OPENAI_MODEL_COVERLETTER:", os.getenv("OPENAI_MODEL_COVERLETTER", "gpt-5-nano"))

Repo root: /Users/jefferysengsy/Documents/GitHub/Spring-2026-DSBA-6010-Group-13-HireMe-AI
OPENAI_API_KEY exists: True
OPENAI_MODEL_RESUME: gpt-5-nano
OPENAI_MODEL_COVERLETTER: gpt-5-nano


In [2]:
from langchain_openai import ChatOpenAI
from models import CandidateProfile, JobPosting
from prompts import build_resume_prompt, build_cover_letter_prompt

resume_template_path = project_dir / "Templates" / "resume_template.md"
cover_template_path = project_dir / "Templates" / "cover_letter_template.md"
candidate_path = project_dir / "candidate_profile.json"

resume_template = resume_template_path.read_text(encoding="utf-8")
cover_template = cover_template_path.read_text(encoding="utf-8")
candidate_json = json.loads(candidate_path.read_text(encoding="utf-8"))

profile = CandidateProfile(**candidate_json)

job_json = {
    "job_title": "Data Engineer II",
    "company_name": "Meridian Technologies",
    "job_description": (
        "This position is responsible for development, consulting and validating of security logs analytics from various security and IT services. "
        "The CW will be working within the Cyber Security Operations Center to assist in maturing and enriching cyber investigations."
    )
}
job = JobPosting(**job_json)

print("Candidate:", profile.name)
print("Target role:", job.job_title, "@", job.company_name)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Candidate: 
Target role: Data Engineer II @ Meridian Technologies


In [6]:
resume_prompt = build_resume_prompt(profile, job, resume_template)
cover_prompt = build_cover_letter_prompt(profile, job, cover_template)

print("Resume prompt preview:\n")
print(resume_prompt[:900])
print("\n---\n")
print("Cover prompt preview:\n")
print(cover_prompt[:900])

Resume prompt preview:


You are an expert resume writer. Tailor the resume to the job description. 

TAILORING RULES:
- Use the template to structure exactly. 
- Prioritize content relevant to the job description. 
- Bullets must be impact/result focused (numbers if present in the data; do not invent).
- Do not invent employers, degrees, dates, certifications, or metrics. 
- If information is missing, omit it. 

RESUME TEMPLATE:
# [Name]
[Email] | [Phone] | [Website]

## Summary
[2-4 sentence summary of professional background, strengths, and target role.]

## Work Experience
### [Job Title] - [Company]
[Month Year] - [Month Year or Present]
- [Impact/result-focused bullet]
- [Impact/result-focused bullet]
- [Impact/result-focused bullet]

### [Job Title] - [Company]
[Month Year] - [Month Year]
- [Impact/result-focused bullet]
- [Impact/result-focused bullet]

## Education
### [Degree] - [School]
[Month Year] 

---

Cover prompt preview:


You are an expert cover letter writer.

TAILO

In [12]:
resume_llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL_RESUME", "gpt-5-nano"),
    temperature=0.2,
)

cover_llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL_COVERLETTER", "gpt-5-nano"),
    temperature=0.2,
)

resume_md = resume_llm.invoke(resume_prompt).content.strip()
cover_md = cover_llm.invoke(cover_prompt).content.strip()

print("Resume output preview:\n")
print(resume_md[:1200])
print("\n---\n")
print("Cover letter output preview:\n")
print(cover_md[:1200])

KeyboardInterrupt: 

In [8]:
out_dir = repo_root / "notebooks" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

resume_out = out_dir / "resume_output.md"
cover_out = out_dir / "cover_letter_output.md"

resume_out.write_text(resume_md, encoding="utf-8")
cover_out.write_text(cover_md, encoding="utf-8")

print("Saved:", resume_out)
print("Saved:", cover_out)

Saved: /Users/jefferysengsy/Documents/GitHub/Spring-2026-DSBA-6010-Group-13-HireMe-AI/notebooks/outputs/resume_output.md
Saved: /Users/jefferysengsy/Documents/GitHub/Spring-2026-DSBA-6010-Group-13-HireMe-AI/notebooks/outputs/cover_letter_output.md


In [16]:
llm = ChatOpenAI(api_key=os.getenv("OPENAI_API_KEY"), model="gpt-5-nano")
print(llm.invoke("Give me reasons not to skip the gym today.").content)


Here are solid reasons not to skip the gym today:

- Mood boost right away: exercise releases endorphins and dopamine, which can lift your mood and reduce stress in minutes.
- More energy, not less: even a short workout can wake you up and sharpen focus for the rest of the day.
- Momentum matters: sticking to a routine builds consistency, which is the smartest way to hit longer-term goals.
- Health perks you feel over time: better heart health, stronger bones, and improved metabolism add up with regular gym time.
- Sleep and recovery: regular exercise tends to improve sleep quality, which makes you feel better tomorrow.
- Confidence and self-efficacy: showing up reinforces you can follow through, which makes future goals feel more doable.
- Mental clarity and resilience: a workout can help you tackle tasks with a clearer head and less anxiety.
- Social and accountability benefits: gym days can add accountability, whether you train with a friend or you’re part of a gym community.
- Inju

In [8]:
from langchain_openai import ChatOpenAI
from tools_llm import TOOLS

llm = ChatOpenAI(model="gpt-5-nano", temperature=0.2)
llm_with_tools = llm.bind_tools(TOOLS)
answer = llm_with_tools.invoke("Extract keywords from this JD: This position is responsible for development, consulting and validating of security logs analytics from various security and IT services. "
        "The CW will be working within the Cyber Security Operations Center to assist in maturing and enriching cyber investigations.")
print("tool_calls:", answer.tool_calls)


tool_calls: [{'name': 'extract_job_keywords', 'args': {'job_description': 'This position is responsible for development, consulting and validating of security logs analytics from various security and IT services. The CW will be working within the Cyber Security Operations Center to assist in maturing and enriching cyber investigations.'}, 'id': 'call_fqwwwMDvjr2ETPeFH6ZkhCMu', 'type': 'tool_call'}]
